In [29]:
from pydantic_ai import Agent
from pydantic_ai.models.openai import OpenAIModel
from pydantic import BaseModel, Field
import os
import logfire
from dotenv import load_dotenv
import nest_asyncio
nest_asyncio.apply()



load_dotenv()


logfire.configure()
model = OpenAIModel('o3-mini')
logfire.instrument_pydantic_ai()



system_prompt = """
You are an amazing SQL query generator. 
You can read between the lines and generate the most efficient SQL query for the user's request.
You will be given a user request in natural language and you will need to generate a valid SQL query.
You will then probably receive a response from the user with some feedback.
You will then need to iterate on the SQL query until the user is satisfied.
if something is unclear, ask the user for more information. 
The clarifications should be in the same language as the user's request and not include SQL language.
"""
agent = Agent(model=model, system_prompt=system_prompt)
result = agent.run_sync(
    user_prompt="I want to know the total sales for each product in the year 2024."
)

print(result)


21:47:49.055 agent run
21:47:49.056   preparing model request params
21:47:49.056   chat o3-mini


Logfire project URL: ]8;id=777695;https://logfire-us.pydantic.dev/yiatridis/playground\https://logfire-us.pydantic.dev/yiatridis/playground]8;;\

AgentRunResult(data="Assuming you have a table called sales with at least these three columns: product, sale_date, and amount, you can use the following query:\n\nSELECT product,\n       SUM(amount) AS total_sales\nFROM sales\nWHERE sale_date BETWEEN '2024-01-01' AND '2024-12-31'\nGROUP BY product;\n\nThis query calculates the sum of sales (amount) for each product within the year 2024. If your table or column names differ, please let me know so we can adjust the query accordingly.")


In [25]:
from rich import print as rprint
from pprint import pprint
rprint(result.data)
pprint(result.new_messages)

pprint(result.all_messages)

Assuming you have a table named "sales" with columns like "product_id", "sale_date" (or similar) and "amount" 
(representing the sale value), you can use a query like:

SELECT 
  product_id, 
  SUM(amount) AS total_sales
FROM sales
WHERE sale_date >= '2024-01-01' 
  AND sale_date < '2025-01-01'
GROUP BY product_id;

This query sums up the "amount" for each product where the "sale_date" falls in the year 2024. Let me know if your 
table or column names are different, or if you need additional adjustments!

<bound method AgentRunResult.new_messages of AgentRunResult(data='Assuming you have a table named "sales" with columns like "product_id", "sale_date" (or similar) and "amount" (representing the sale value), you can use a query like:\n\nSELECT \n  product_id, \n  SUM(amount) AS total_sales\nFROM sales\nWHERE sale_date >= \'2024-01-01\' \n  AND sale_date < \'2025-01-01\'\nGROUP BY product_id;\n\nThis query sums up the "amount" for each product where the "sale_date" falls in the year 2024. Let me know if your table or column names are different, or if you need additional adjustments!')>
<bound method AgentRunResult.all_messages of AgentRunResult(data='Assuming you have a table named "sales" with columns like "product_id", "sale_date" (or similar) and "amount" (representing the sale value), you can use a query like:\n\nSELECT \n  product_id, \n  SUM(amount) AS total_sales\nFROM sales\nWHERE sale_date >= \'2024-01-01\' \n  AND sale_date < \'2025-01-01\'\nGROUP BY product_id;\n\nThis query s

In [26]:
result = agent.run_sync(
    user_prompt="I want to add 2023 results as well",
    message_history=result.new_messages()
)

print(result)


21:45:57.906 agent run
21:45:57.907   preparing model request params
21:45:57.908   chat o3-mini
AgentRunResult(data="Assuming you'd like to see separate totals for each product for both 2023 and 2024—with a row per product per year—you can modify the query to extract the year from the sale date. For example, if you're using MySQL, you might write:\n\nSELECT \n  product_id, \n  YEAR(sale_date) AS sale_year,\n  SUM(amount) AS total_sales\nFROM sales\nWHERE sale_date >= '2023-01-01'\n  AND sale_date < '2025-01-01'\nGROUP BY product_id, YEAR(sale_date);\n\nThis returns the total sales for each product with a breakdown by year (2023 and 2024).\n\nIf instead you’d like the results to show one row per product with separate columns for 2023 and 2024 sales, you can use conditional aggregation:\n\nSELECT \n  product_id,\n  SUM(CASE WHEN sale_date >= '2023-01-01' AND sale_date < '2024-01-01' THEN amount ELSE 0 END) AS total_sales_2023,\n  SUM(CASE WHEN sale_date >= '2024-01-01' AND sale_date < '

Currently retrying 1 failed export(s)


In [28]:
pprint(result.data)
pprint(result.new_messages)
pprint(result.all_messages)

("Assuming you'd like to see separate totals for each product for both 2023 "
 'and 2024—with a row per product per year—you can modify the query to extract '
 "the year from the sale date. For example, if you're using MySQL, you might "
 'write:\n'
 '\n'
 'SELECT \n'
 '  product_id, \n'
 '  YEAR(sale_date) AS sale_year,\n'
 '  SUM(amount) AS total_sales\n'
 'FROM sales\n'
 "WHERE sale_date >= '2023-01-01'\n"
 "  AND sale_date < '2025-01-01'\n"
 'GROUP BY product_id, YEAR(sale_date);\n'
 '\n'
 'This returns the total sales for each product with a breakdown by year (2023 '
 'and 2024).\n'
 '\n'
 'If instead you’d like the results to show one row per product with separate '
 'columns for 2023 and 2024 sales, you can use conditional aggregation:\n'
 '\n'
 'SELECT \n'
 '  product_id,\n'
 "  SUM(CASE WHEN sale_date >= '2023-01-01' AND sale_date < '2024-01-01' THEN "
 'amount ELSE 0 END) AS total_sales_2023,\n'
 "  SUM(CASE WHEN sale_date >= '2024-01-01' AND sale_date < '2025-01-01' THEN "
 